# 12주차 | 채용·HR 데이터와 AI 윤리
**오늘의 질문:** 같은 역량을 가진 사례에 AI가 다른 결과를 주면 공정할까?

CPU만 사용. 실제 채용, 실제 지원자 평가에 사용 금지. 고용24는 채용공고 정보이며 지원자·합격자 데이터가 아니다. API 형식 모의 XML 6건과 인위적 감사 사례 240건을 사용한다. 실서비스 호출은 선택 실습이다. 파일 → Drive에 사본 저장 후 위에서부터 실행한다.

## 0. 준비
라이브러리 불러오기. 데이터는 현재 폴더, GitHub 순으로 읽는다. 실패할 경우 Drive의 세 파일을 다운로드해 Colab 왼쪽 파일 창에 업로드하고 다시 실행한다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from urllib.request import urlopen
import zipfile
BASE = 'https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week12/data/'
def load_file(name):
    for p in [Path(name), Path('data')/name, Path('../data')/name]:
        if p.is_file(): return p.read_bytes()
    try:
        with urlopen(BASE+name, timeout=20) as r: return r.read()
    except Exception as e:
        raise RuntimeError('Drive 데이터 파일을 다운로드하여 Colab 파일 창에 업로드한 뒤 재실행하세요: '+name) from e
from io import BytesIO
out=Path('week12_results'); out.mkdir(exist_ok=True)
print('Ready: CPU only')

## 1. 공고 데이터 읽기 (XML → 표)
`wanted`는 공고 1건, 자식 태그는 해당 공고의 항목이다. `practice_skills`는 교육자가 만든 별도 연습용 요구사항이다. 공식 API에서 이 값이 반환된다는 뜻이 아니다.

In [ ]:
xml=ET.fromstring(load_file('work24_demo.xml'))
jobs=pd.DataFrame([{k:n.findtext(k) for k in ['wantedAuthNo','title','region','career','minEdubg']} for n in xml.findall('wanted')])
requirements=pd.read_csv(BytesIO(load_file('job_requirements.csv')))
jobs=jobs.merge(requirements[['job_id','practice_skills']],left_on='wantedAuthNo',right_on='job_id')
print(jobs.to_string(index=False)); print('CP1:',len(jobs),'postings')
assert len(jobs)==6

## 2. 스크리닝 기준을 정하는 활동
가상 J01 데이터 분석 보조에 대해 Python·SQL 역량 확인과 과제 점수 65 이상이라는 규칙을 정했다. 시험용 기준이며 실제 업무 능력 판단이나 채용 기준이 아니다. 인위적 역사적 AI 점수는 사전에 만들어진 감사 대상이지 실제 모델 예측이 아니다.

In [ ]:
df=pd.read_csv(BytesIO(load_file('synthetic_audit.csv')))
print(df.head(6).to_string(index=False)); print('CP2:',df.shape)
assert df.shape==(240,9) and df.isna().sum().sum()==0
assert df.groupby('pair_id').size().eq(2).all()

## 3. 수작업 기준과 기존 AI 결과 재현
`manual`은 교실에서 합의한 기준 충족 여부. AI 점수 0.65 이상을 과거 시스템의 모의 양성 결과로 해석한다. 모두 합성 사례에 대한 감사이며 실제 사람을 선별하지 않는다.

In [ ]:
df['manual']=((df.python_skill==1)&(df.sql_skill==1)&(df.task_score>=65)).astype(int)
df['ai_flag']=(df.historical_ai_score>=0.65).astype(int)
compare=pd.crosstab(df.manual,df.ai_flag,rownames=['manual'],colnames=['AI'])
print(compare); print('agreement:',round((df.manual==df.ai_flag).mean(),3))
compare.to_csv(out/'comparison.csv')

## 4. 집단별 감사
분모는 각 집단 사례 수. 양성률=양성 사례/전체, 기준 충족 사례 중 AI 양성률(TPR)=기준 충족·AI 양성/기준 충족. A/B는 임의로 부여한 합성 감사 그룹으로 실제 속성이 아니다.

In [ ]:
summary=df.groupby('audit_group').agg(n=('case_id','size'),manual_rate=('manual','mean'),ai_rate=('ai_flag','mean'))
summary['tpr']=df[df.reference==1].groupby('audit_group').ai_flag.mean()
summary['gap_from_manual']=summary.ai_rate-summary.manual_rate
print(summary.round(3)); print('AI rate gap (A-B):',round(summary.loc['A','ai_rate']-summary.loc['B','ai_rate'],3))
summary.to_csv(out/'group_audit.csv')
ax=summary[['manual_rate','ai_rate','tpr']].plot.bar(rot=0,ylim=(0,1),color=['#64748b','#2563eb','#f59e0b'])
ax.set(ylabel='Rate',title='Synthetic screening audit'); ax.legend(loc='upper right')
plt.tight_layout(); plt.savefig(out/'group_audit.png',dpi=140); plt.show()

## 5. 짝 비교와 민감도 분석
같은 역량의 A/B 두 사례를 비교한다. 점수 기준을 바꾸면 결과 차이가 달라지는지 관찰한다. 기준을 조정해 실제 채용에 적용하는 것이 아니라 차이가 왜 발생하는지 감사하는 활동이다.

In [ ]:
pairs=df.pivot(index='pair_id',columns='audit_group',values='ai_flag')
print('Different output pairs:',int((pairs.A!=pairs.B).sum()),'/',len(pairs))
rates=[]
for t in [0.50,0.60,0.65,0.70,0.80]:
    s=df.assign(flag=df.historical_ai_score>=t).groupby('audit_group').flag.mean()
    rates.append({'threshold':t,'A':s.A,'B':s.B,'gap':s.A-s.B})
rates=pd.DataFrame(rates); print(rates.round(3)); rates.to_csv(out/'threshold_audit.csv',index=False)
ax=rates.plot(x='threshold',y=['A','B'],marker='o',ylim=(0,1),title='Threshold sensitivity')
ax.set_ylabel('Positive rate'); plt.tight_layout(); plt.savefig(out/'threshold_audit.png',dpi=140); plt.show()

## 6. 결과 기록 및 제출
텍스트 셀을 더블 클릭해 답을 작성한다.
- Q1. 공고 정보와 지원자 데이터의 차이는?
- Q2. 일치율, 집단별 양성률 차이, 불일치 쌍 수는?
- Q3. 같은 역량의 결과가 다른 이유와 이 자료의 한계는?
- Q4. 실제 서비스 도입 전에 어떤 검토와 사람의 감독이 필요한가?

나의 답: (작성)

In [ ]:
df.to_csv(out/'case_audit.csv',index=False)
with zipfile.ZipFile('Week12_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out.iterdir()): z.write(p,p.name)
print('CP5: Week12_results.zip',sorted(p.name for p in out.iterdir()))
# Colab 왼쪽 파일 창에서 ZIP 다운로드; ipynb는 파일 > 다운로드

## 선택: 실제 고용24 API 연결 (교수자 시연용, 필수 실행 아님)
[공식 API 명세 및 신청](https://www.work24.go.kr/cm/e/a/0110/selectOpenApiSvcInfo.do?fullApiSvcId=000000000000000000000000000000)에서 인증키를 신청한다. 키는 코드나 공유 파일에 저장하지 말고 Colab 보안 비밀 또는 입력창을 이용한다. 응답은 XML이며 실제 공고만 제공된다. 키가 없다면 1단계의 모의 XML로 충분히 수행할 수 있다. 아래 코드는 설명용으로 별도 셀에서 실행한다.
```python
import requests, getpass
key = getpass.getpass('개인 인증키: ')
url = 'https://www.work24.go.kr/cm/openApi/call/wk/callOpenApiSvcInfo210L01.do'
r = requests.get(url, params={'authKey':key,'callTp':'L','returnType':'XML','startPage':1,'display':10}, timeout=20)
r.raise_for_status()
root = ET.fromstring(r.content)
print('공고 건수:',len(root.findall('wanted')))
```
실제 응답의 재배포 조건을 확인하고 원자료와 합성 자료는 섞지 않는다.